# Sprint 4: Ensembles Predictivos (Voting, Bagging, Stacking)

Entrenamiento de modelos compuestos.

In [1]:
import sys
sys.path.append("../src")
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

df = pd.read_csv("../data/processed/features_data.csv")
X = df.drop(columns=['attrition'])
y = df['attrition']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preproc = joblib.load("../models/preprocessing_pipeline.pkl")

# Cargar hiperparámetros optimizados
params_rf = joblib.load("../models/tuned_rf_optuna.pkl")
params_svm = joblib.load("../models/tuned_svm_optuna.pkl")
params_lr = joblib.load("../models/tuned_lr_optuna.pkl")
params_knn = joblib.load("../models/tuned_knn_optuna.pkl")

estimators = [
    ('rf', RandomForestClassifier(**params_rf)),
    ('svm', SVC(**params_svm)),
    ('lr', LogisticRegression(**params_lr)),
    ('knn', KNeighborsClassifier(**params_knn))
]

stack_clf = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression())
stack_pipeline = ImbPipeline([
    ('preproc', preproc),
    ('smote', SMOTE(random_state=42)),
    ('clf', stack_clf)
])
stack_pipeline.fit(X_train, y_train)
joblib.dump(stack_pipeline, "../models/final_model.pkl")
print("Stacking final_model.pkl entrenado y guardado.")

Stacking final_model.pkl entrenado y guardado.
